In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load PDF
loader = PyPDFLoader("short_story.pdf")
docs = loader.load()

# Split text
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Embeddings (local)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

C:\Users\erinx\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\erinx\AppData\Local\Temp\ipykernel_19780\900284363.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1416.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | 

In [2]:
from langchain_openai import OpenAI

llm = OpenAI(
    base_url="http://127.0.0.1:1015/v1",
    api_key="lm-studio",
    model="qwen/qwen3-vl-4b",
    temperature=0.2
)

In [3]:
from langchain_community.vectorstores import FAISS
import os

if not os.path.exists("faiss_index"):
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local("faiss_index")
else:
    vectorstore = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def retrieve_docs(query):
    if isinstance(query, dict):
        query = query.get("query") or query.get("action_input") or str(query)
    
    print(f"--- Document Search Tool Triggered with Query: {query} ---")
    
    try:
        found_docs = retriever.invoke(query)
        
        if not found_docs:
            return "No relevant information found in the document."
            
        context = "\n\n".join([doc.page_content for doc in found_docs])
        return context
        
    except Exception as e:
        return f"Error during retrieval: {str(e)}"

In [4]:
from docx import Document
import os
from datetime import datetime

def create_word_doc(content, filename="generated.docx", folder="generated_docs"):
    os.makedirs(folder, exist_ok=True)

    filepath = os.path.join(folder, filename)

    doc = Document()

    # Title
    doc.add_heading("Generated Document", 0)

    # Content (split into paragraphs)
    for line in content.split("\n"):
        doc.add_paragraph(line)

    doc.save(filepath)

    return f"Document saved at: {filepath}"

In [5]:
from langchain_core.tools import Tool

def word_tool(query):
    prompt = f"""
Create a well-structured document:

{query}

    Format:
    - Title
    - Sections
    - Bullet points if needed
    """

    content = llm.invoke(prompt).content  # FIXED

    filename = f"doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
    return create_word_doc(content, filename)

word_tool_instance = Tool(
    name="Word Document Generator",
    func=word_tool,
    description="Creates a Word document from a user request"
)

In [6]:
import math

def tool_calculator(query):     # This tool allows the agent to solve arithmetic expressions
    try:
        result = eval(query, {"__builtins__": None}, {"sqrt": math.sqrt})
        return str(result)
    except:
        return "Invalid arithmetic expression"

calculator_tool = Tool(
    name="Calculator",
    func=tool_calculator,
    description="Solve arithmetic problems like addition, subtraction, multiplication, and division"
)

In [7]:
import wikipedia

def tool_wikipedia(query):
    try:
        return wikipedia.summary(query, sentences=5)
    except Exception as e:
        return f"Error: {str(e)}"

wikipedia_tool = Tool(
    name="Wikipedia Extractor",
    func=tool_wikipedia,
    description="Search Wikipedia and return a summary of a topic. " \
    "Use this tool whenever the user asks for information 'according to Wikipedia "
    "or asks for general knowledge about a topic like science, history, technology, etc."
)

In [8]:
def wiki_doc_creator(query):          # This tool extracts Wikipedia info and creates a Word document from it
    try:
        summary = wikipedia.summary(query, sentences=10)
    except:
        summary = "Wikipedia article not found."

    prompt = f"""
    Create a structured document using this information:

    {summary}

    Format:
    - Title
    - Sections
    - Bullet points if needed
    """

    content = llm.invoke(prompt).content

    filename = f"wiki_doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"

    return create_word_doc(content, filename)

wikipedia_doc_tool = Tool(
    name="Wikipedia Document Creator",
    func=wiki_doc_creator,
    description="Creates a Word document using information extracted from Wikipedia"
)

In [9]:
from langchain_classic.agents import Tool, initialize_agent, AgentType
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="input",
    return_messages=True
)

doc_retrieval_tool = Tool(
        name="Document Search",
        func=retrieve_docs,
        description="Use this to answer questions from PDFs or documents"
    )

tools = [
    word_tool_instance,
    doc_retrieval_tool,
    calculator_tool,
    wikipedia_tool,
    wikipedia_doc_tool
]


agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,  
    verbose=True,
    return_intermediate_steps=False,
    handle_parsing_errors=True,
    memory=memory,
    max_iterations=5,  # prevent infinite loops
    early_stopping_method="generate",
    agent_kwargs={
        "system_message": """
You are an intelligent AI assistant with access to tools.

AVAILABLE TOOLS:
- Document Search → for answering questions from uploaded PDFs or documents
- Word Document Generator → for creating .docx files
- Calculator → solve arithmetic problems
- Wikipedia Extractor → extract information from Wikipedia
- Wikipedia Document Creator → create Word documents using Wikipedia data

DECISION RULES:
1. If the question is about documents or requires context → use Document Search
2. If the user asks to create/generate/write a file → use Word Document Generator
3. If the question is general knowledge → give your direct answer and give some information from wikipedia if needed
4. If the user asks for calculations → use Calculator
5. If the user asks for Wikipedia information → use Wikipedia Extractor
6. If the user asks to create a document from Wikipedia → use Wikipedia Document Creator
7. Always provide a final answer after using tools, make your answer clean and clear,and summarize information when needed.
8. If you use a tool, explain your reasoning in the final answer.

IMPORTANT:
- Do NOT explain your reasoning
- Be concise and precise
"""
    }
)



C:\Users\erinx\AppData\Local\Temp\ipykernel_19780\1833808706.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(
C:\Users\erinx\AppData\Local\Temp\ipykernel_19780\1833808706.py:25: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


In [10]:
import re

def is_calculation(query):
    return bool(re.fullmatch(r"[0-9+\-*/(). sqrt]+", query.lower().strip()))

def answer_from_document(query):
    context = retrieve_docs(query)
    prompt = f"""
You are answering questions about the PDF/document.
Use ONLY the document context below.
If the answer is not clearly supported by the context, say: "I could not find that in the document."
Do not add outside facts. Do not guess.

Question: {query}

Document context:
{context}
"""
    return llm_text(prompt)

def route_query(query):
    q = query.lower().strip()

    if is_calculation(q):
        return calculator_tool.run(query)

    if "wikipedia" in q and any(word in q for word in ["document", "doc", "file", "create", "generate"]):
        return wikipedia_doc_tool.run(query.replace("wikipedia", "").strip())

    if any(word in q for word in ["docx", "word file", "create a file", "generate a file", "create document", "generate document"]):
        return word_tool_instance.run(query)

    if "wikipedia" in q:
        return wikipedia_tool.run(query.replace("wikipedia", "").strip())

    # Default to the PDF/document so answers stay grounded instead of hallucinating.
    return answer_from_document(query)

def ask_agent(query):
    query = query.strip()
    if not query:
        return "Please enter a question."
    return str(route_query(query))

while True:
    query = input("Ask a question (type 'exit' to quit): ")

    if query.strip().lower() == "exit":
        print("Exiting...")
        print("Thank you for using the AI assistant. Goodbye!")
        break

    try:
        print("User Input:", query)
        output = ask_agent(query)
    except Exception as e:
        output = f"Error while running the assistant: {e}"

    print("-" * 150)
    print("AI Response:")
    print(">> " + output)
    print("-" * 150)
    print("\n" + "=" * 150 + "\n")

User Input: summarize the short story pdf
--- Document Search Tool Triggered with Query: summarize the short story pdf ---
------------------------------------------------------------------------------------------------------------------------------------------------------
AI Response:
>> Error while running the assistant: name 'llm_text' is not defined
------------------------------------------------------------------------------------------------------------------------------------------------------


User Input: summarize story
--- Document Search Tool Triggered with Query: summarize story ---
------------------------------------------------------------------------------------------------------------------------------------------------------
AI Response:
>> Error while running the assistant: name 'llm_text' is not defined
------------------------------------------------------------------------------------------------------------------------------------------------------


Exiting...